In [ ]:
#!/usr/bin/env bash
# install.sh — set up and manage the credential daemon
# Usage:
#   sudo bash install.sh install
#   sudo bash install.sh uninstall
#   sudo bash install.sh status
#   sudo bash install.sh logs

set -euo pipefail

SERVICE_NAME="credential-daemon"
SERVICE_FILE="credential-daemon.service"
INSTALL_DIR="/opt/credential-daemon"
DATA_DIR="/var/lib/credential-daemon"
LOG_DIR="/var/log/credential-daemon"
KEY_DIR="/etc/credential-daemon"
DAEMON_USER="creddaemon"

check_root() {
    [[ "$EUID" -ne 0 ]] && echo "Run as root (sudo)." && exit 1
}

cmd_install() {
    check_root
    echo "── Creating user ────────────────────────────────────────"
    id "$DAEMON_USER" &>/dev/null || useradd -r -s /bin/false "$DAEMON_USER"

    echo "── Creating directories ──────────────────────────────────"
    mkdir -p "$INSTALL_DIR" "$DATA_DIR" "$LOG_DIR" "$KEY_DIR"

    echo "── Copying files ─────────────────────────────────────────"
    cp ssh_daemon.py "$INSTALL_DIR/ssh_daemon.py"
    cp "$SERVICE_FILE" "/etc/systemd/system/$SERVICE_FILE"

    echo "── Setting permissions ───────────────────────────────────"
    chown -R "$DAEMON_USER:$DAEMON_USER" "$DATA_DIR" "$LOG_DIR" "$KEY_DIR"
    chmod 750 "$DATA_DIR" "$LOG_DIR" "$KEY_DIR"
    chmod 644 "/etc/systemd/system/$SERVICE_FILE"

    echo "── Enabling service ──────────────────────────────────────"
    systemctl daemon-reload
    systemctl enable "$SERVICE_NAME"
    systemctl start  "$SERVICE_NAME"

    echo ""
    echo "✓ Installed. Status:"
    systemctl status "$SERVICE_NAME" --no-pager
    echo ""
    echo "Port 9922 must be open for WAN access:"
    echo "  sudo ufw allow 9922/tcp"
    echo "  # or iptables:"
    echo "  sudo iptables -I INPUT -p tcp --dport 9922 -j ACCEPT"
}

cmd_uninstall() {
    check_root
    echo "── Stopping and disabling ────────────────────────────────"
    systemctl stop    "$SERVICE_NAME" || true
    systemctl disable "$SERVICE_NAME" || true
    rm -f "/etc/systemd/system/$SERVICE_FILE"
    systemctl daemon-reload
    echo "── Removing files ────────────────────────────────────────"
    rm -rf "$INSTALL_DIR"
    echo "Note: data/logs/keys left intact at $DATA_DIR $LOG_DIR $KEY_DIR"
    echo "Remove manually if needed."
}

cmd_status() {
    systemctl status "$SERVICE_NAME" --no-pager
}

cmd_logs() {
    journalctl -u "$SERVICE_NAME" -f --no-pager
}

case "${1:-help}" in
    install)   cmd_install   ;;
    uninstall) cmd_uninstall ;;
    status)    cmd_status    ;;
    logs)      cmd_logs      ;;
    *)
        echo "Usage: sudo bash install.sh [install|uninstall|status|logs]"
        ;;
esac